# Repeat Purchase Prediction

## 02. Data Preprocessing

### Notebook purpose

This notebook prepares the data for the predictive modelling stage of the
Repeat Purchase Prediction project.

The preprocessing aims to:

- Clean missing, invalid and anomalous values identified during the EDA.
- Integrate additional customer information through `customer_id`.
- Create features that may help predict repeat purchases within 90 days.
- Transform categorical variables into a format suitable for modelling.
- Apply scaling where appropriate.
- Define and implement a feature selection strategy.
- Produce consistent training and test datasets for the modelling stage.

All preprocessing decisions are based on the findings from the EDA and are
implemented without using information from the test target.

## 1. Environment Setup and Dataset Size Assessment

We begin by importing the required libraries and locating the data folder.
Before loading the datasets, we check their file sizes to assess the resources
needed for exploration.

In [27]:
# Handle file paths without relying on computer-specific locations.
from pathlib import Path

# Locate the project from the current folder or one of its parent folders.
PROJECT_ROOT = next(
    (
        folder
        for folder in [Path.cwd(), *Path.cwd().parents]
        if (folder / "data").is_dir()
        and (folder / "notebooks").is_dir()
    ),
    None,
)

# Give a clear error if the expected project structure cannot be found.
if PROJECT_ROOT is None:
    raise FileNotFoundError(
        "Cannot locate the project. Open the notebook within the repository."
    )

DATA_DIR = PROJECT_ROOT / "data"

# Check the expected datasets without loading their contents.
DATA_FILES = [
    "train.csv",
    "test.csv",
    "clientes.csv",
    "sample_submission.csv",
]

total_size_mib = 0

for filename in DATA_FILES:
    file_path = DATA_DIR / filename

    if not file_path.is_file():
        raise FileNotFoundError(f"Missing dataset: {file_path}")

    # Convert file size from bytes to mebibytes (MiB).
    size_mib = file_path.stat().st_size / (1024 ** 2)
    total_size_mib += size_mib

    print(f"{filename}: {size_mib:,.2f} MiB")

print(f"\nTotal size: {total_size_mib:,.2f} MiB")

train.csv: 2.39 MiB
test.csv: 0.36 MiB
clientes.csv: 0.34 MiB
sample_submission.csv: 0.01 MiB

Total size: 3.11 MiB


### 1.1. Imports and Display Settings

Import the libraries used throughout the analysis and configure display
settings to keep tables readable and plots consistent.

In [28]:
# Data manipulation and numerical calculations.
import pandas as pd
import numpy as np

# Charts for exploring distributions and relationships.
import matplotlib.pyplot as plt
import seaborn as sns


# Display multiple tables explicitly within a notebook cell.
from IPython.display import display

# Show more columns and longer text when inspecting DataFrames.
pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 100)
pd.set_option("display.max_colwidth", 100)

# Format displayed numbers without changing the underlying values.
pd.set_option("display.float_format", "{:,.3f}".format)

# Use consistent styling and colours across plots.
sns.set_theme(style="whitegrid", palette="colorblind")
plt.rcParams["figure.figsize"] = (10, 5)
plt.rcParams["axes.titlesize"] = 14
plt.rcParams["axes.labelsize"] = 11

### 1.3. Data Loading

Load the datasets.

In [29]:
# Load the datasets into separate DataFrames.
train = pd.read_csv(DATA_DIR / "train.csv")
test = pd.read_csv(DATA_DIR / "test.csv")
clientes = pd.read_csv(DATA_DIR / "clientes.csv")
sample_submission = pd.read_csv(DATA_DIR / "sample_submission.csv")

# Group the datasets to inspect them consistently.
datasets = {
    "train": train,
    "test": test,
    "clientes": clientes,
    "sample_submission": sample_submission,
}


### 1.4 Dataset Overview

Before starting the preprocessing steps, we verify the dimensions and basic structure of the loaded datasets. This provides a final check that the data has been loaded correctly and gives an overview of the information available in each dataset.

In [30]:
# Display the dimensions of each dataset.

for name, df in datasets.items():
    print(f"{name}: {df.shape[0]:,} rows × {df.shape[1]:,} columns")
    

# Inspect the data types of the training dataset.
train.dtypes

# Count missing values in each dataset.

for name, df in datasets.items():
    print(f"\n{name}")
    display(df.isna().sum().sort_values(ascending=False).head(10))

train: 6,534 rows × 39 columns
test: 986 rows × 38 columns
clientes: 1,143 rows × 13 columns
sample_submission: 986 rows × 2 columns

train


prior_std_gap_days              1478
prior_mean_gap_days             1308
prior_std_purchase_value        1113
prior_mean_purchase_value        862
days_since_previous_purchase     861
prior_max_purchase_value         668
prior_min_purchase_value         668
transport_method                 261
warehouse                        260
payment_type                     260
dtype: int64


test


prior_mean_gap_days             76
prior_std_gap_days              69
prior_mean_purchase_value       52
days_since_previous_purchase    52
prior_std_purchase_value        46
warehouse                       30
transport_method                30
salesperson                     30
payment_type                    30
spend_365d                      30
dtype: int64


clientes


customer_type        1143
street                598
island                578
city                  546
postal_code           170
municipality          159
commercial_region     114
postal_area            71
address                 6
customer_id             0
dtype: int64


sample_submission


ID                     0
repeat_purchase_90d    0
dtype: int64

## 2. Preprocessing Strategy

The EDA identified several issues that need to be addressed before modelling.
These include missing values, observations without purchase information,
invalid values in variables measuring time between purchases, extreme spending
values and variables with little or no useful information.

The preprocessing steps will therefore be based on the following decisions:

| Issue identified in EDA | Preprocessing approach |
|---|---|
| 65 observations without purchase information | Remove these observations |
| Missing values in historical variables | Compare suitable imputation strategies |
| Negative `-4` values in time-gap variables | Treat as invalid values and investigate as missing |
| Extreme spending values | Investigate and determine an appropriate treatment |
| Completely missing variables | Remove variables with no observed information |
| Customer information available in `clientes.csv` | Merge using `customer_id` |
| Date stored as text | Convert to datetime and create relevant date features |
| Categorical variables | Encode variables according to their characteristics |
| `random_noise` | Assess its usefulness before deciding whether to remove it |
| Similar historical variables | Retain initially and consider them during feature selection |


The preprocessing will be applied consistently to the training and test datasets. However, any transformation that requires parameters to be estimated from the data will be fitted using the training data only and then applied to validation and test data. This prevents information from unseen data from influencing the preprocessing or model development.

## 3. Data Cleaning

### 3.1 Remove Observations Without Purchase Information

In [31]:
# Create copies for preprocessing so that the original datasets remain unchanged.

train_prep = train.copy()
test_prep = test.copy()
clientes_prep = clientes.copy()

# Check
print("Original train shape:", train.shape)
print("Preprocessing train shape:", train_prep.shape)

print("Original test shape:", test.shape)
print("Preprocessing test shape:", test_prep.shape)

print("Original clientes shape:", clientes.shape)
print("Preprocessing clientes shape:", clientes_prep.shape)

Original train shape: (6534, 39)
Preprocessing train shape: (6534, 39)
Original test shape: (986, 38)
Preprocessing test shape: (986, 38)
Original clientes shape: (1143, 13)
Preprocessing clientes shape: (1143, 13)


In [32]:
# Identify purchase-related columns.
purchase_columns = [
    col for col in train_prep.columns
    if col not in ["ID", "customer_id", "repeat_purchase_90d"]
]

# Identify observations with no purchase information at all.
no_purchase_train = train_prep[purchase_columns].isna().all(axis=1)

print("Observations without purchase information:", no_purchase_train.sum())

Observations without purchase information: 65


In [33]:
# Remove observations without any purchase information.
train_prep = train_prep.loc[~no_purchase_train].copy()

print("Training shape after removing incomplete observations:", train_prep.shape)

# Verify that no observations with all purchase fields missing remain.
print(
    "Remaining observations without purchase information:",
    train_prep[purchase_columns].isna().all(axis=1).sum()
)

Training shape after removing incomplete observations: (6469, 39)
Remaining observations without purchase information: 0


In [34]:
no_purchase_test = test_prep[purchase_columns].isna().all(axis=1)

print("Test observations without purchase information:", no_purchase_test.sum())

Test observations without purchase information: 0


#### Observation

The 65 observations removed above contained no purchase-related information, with all purchase fields missing. Since these observations do not provide the transaction information required to construct meaningful predictors for the repeat-purchase task, they were excluded from the training data. Other observations with partial missingness are retained and will be investigated separately rather than removed automatically.

### 3.2 Handle Invalid Time-Gap Values

The EDA identified `-4` values in variables measuring time between purchases. Since these variables represent elapsed time, negative values are not consistent with their intended meaning. These values will therefore be treated as invalid and converted to missing values (`NaN`), allowing them to be handled together with the other missing values in the following preprocessing step.

In [35]:
# Time-gap variables identified during the EDA.
time_gap_columns = [
    "days_since_previous_purchase",
    "prior_mean_gap_days",
    "prior_std_gap_days"
]

In [36]:
# Count invalid -4 values in the training and test datasets.

for col in time_gap_columns:
    print(f"{col}:")
    print(f"  Train: {(train_prep[col] == -4).sum()}")
    print(f"  Test:  {(test_prep[col] == -4).sum()}")
    

days_since_previous_purchase:
  Train: 16
  Test:  2
prior_mean_gap_days:
  Train: 16
  Test:  2
prior_std_gap_days:
  Train: 0
  Test:  0


In [38]:
# Replace invalid negative time-gap values with missing values.

for col in time_gap_columns:
    train_prep.loc[train_prep[col] == -4, col] = np.nan
    test_prep.loc[test_prep[col] == -4, col] = np.nan
    
# Verify that no -4 values remain in the time-gap variables.

for col in time_gap_columns:
    print(f"{col}:")
    print(f"  Train -4 values remaining: {(train_prep[col] == -4).sum()}")
    print(f"  Test -4 values remaining:  {(test_prep[col] == -4).sum()}")

days_since_previous_purchase:
  Train -4 values remaining: 0
  Test -4 values remaining:  0
prior_mean_gap_days:
  Train -4 values remaining: 0
  Test -4 values remaining:  0
prior_std_gap_days:
  Train -4 values remaining: 0
  Test -4 values remaining:  0


#### Observation

The invalid `-4` values were converted to `NaN` rather than replaced by a numerical value. This preserves the distinction between an invalid observation and a genuine zero time gap. The resulting missing values will be addressed together with the remaining missing data after their patterns and possible causes have been investigated.

### 3.3 Handle Missing Values

The EDA showed that missing values occur across several purchase and historical variables. Rather than removing all observations with missing values or applying the same imputation method to every variable, the missingness pattern is first investigated. In particular, historical variables are compared with the customer's previous purchase history to determine whether some missing values have a meaningful structural explanation.

In [39]:
# Calculate the number and percentage of missing values in the training data.

missing_summary = pd.DataFrame({
    "missing_count": train_prep.isna().sum(),
    "missing_percentage": train_prep.isna().mean() * 100
})

missing_summary = (
    missing_summary[missing_summary["missing_count"] > 0]
    .sort_values("missing_count", ascending=False)
)

display(missing_summary)

,missing_count,missing_percentage
prior_std_gap_days,1413,21.843
prior_mean_gap_days,1259,19.462
prior_std_purchase_value,1048,16.200
days_since_previous_purchase,812,12.552
prior_mean_purchase_value,797,12.320
prior_min_purchase_value,603,9.321
prior_max_purchase_value,603,9.321
transport_method,196,3.030
spend_365d,195,3.014
payment_type,195,3.014


In [40]:
# Compare previous purchase history for observations with and without
# missing values in the historical variables.

historical_columns = [
    "prior_purchase_count",
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "prior_mean_gap_days",
    "prior_std_gap_days"
]

missing_history = pd.DataFrame({
    "missing_count": train_prep[historical_columns].isna().sum(),
    "missing_percentage": train_prep[historical_columns].isna().mean() * 100
})

display(missing_history)

,missing_count,missing_percentage
prior_purchase_count,0,0.000
prior_total_spend,0,0.000
prior_mean_purchase_value,797,12.320
prior_std_purchase_value,1048,16.200
prior_min_purchase_value,603,9.321
prior_max_purchase_value,603,9.321
prior_mean_gap_days,1259,19.462
prior_std_gap_days,1413,21.843


In [41]:
# Check previous purchase counts for observations with missing historical values.

for col in historical_columns[1:]:
    missing_rows = train_prep[train_prep[col].isna()]

    print(f"\n{col}")
    print(
        "Number of missing observations:",
        len(missing_rows)
    )
    print(
        "Previous purchase count - median:",
        missing_rows["prior_purchase_count"].median()
    )
    print(
        "Previous purchase count - minimum:",
        missing_rows["prior_purchase_count"].min()
    )
    print(
        "Previous purchase count - maximum:",
        missing_rows["prior_purchase_count"].max()
    )


prior_total_spend
Number of missing observations: 0
Previous purchase count - median: nan
Previous purchase count - minimum: nan
Previous purchase count - maximum: nan

prior_mean_purchase_value
Number of missing observations: 797
Previous purchase count - median: 0.0
Previous purchase count - minimum: 0.0
Previous purchase count - maximum: 242.0

prior_std_purchase_value
Number of missing observations: 1048
Previous purchase count - median: 0.0
Previous purchase count - minimum: 0.0
Previous purchase count - maximum: 1.0

prior_min_purchase_value
Number of missing observations: 603
Previous purchase count - median: 0.0
Previous purchase count - minimum: 0.0
Previous purchase count - maximum: 0.0

prior_max_purchase_value
Number of missing observations: 603
Previous purchase count - median: 0.0
Previous purchase count - minimum: 0.0
Previous purchase count - maximum: 0.0

prior_mean_gap_days
Number of missing observations: 1259
Previous purchase count - median: 1.0
Previous purchase c

In [42]:
# Summarise missing historical values according to previous purchase count.

purchase_history_missingness = []

for count, group in train_prep.groupby("prior_purchase_count", dropna=False):
    purchase_history_missingness.append({
        "prior_purchase_count": count,
        "observations": len(group),
        "missing_mean_purchase_value": group["prior_mean_purchase_value"].isna().sum(),
        "missing_std_purchase_value": group["prior_std_purchase_value"].isna().sum(),
        "missing_mean_gap_days": group["prior_mean_gap_days"].isna().sum(),
        "missing_std_gap_days": group["prior_std_gap_days"].isna().sum()
    })

purchase_history_missingness = pd.DataFrame(purchase_history_missingness)

display(purchase_history_missingness.head(15))

,prior_purchase_count,observations,missing_mean_purchase_value,missing_std_purchase_value,missing_mean_gap_days,missing_std_gap_days
0,0.000,603,603,603,603,603
1,1.000,445,20,445,445,445
2,2.000,365,11,0,25,365
3,3.000,319,14,0,17,0
4,4.000,270,11,0,7,0
5,5.000,248,7,0,8,0
6,6.000,222,6,0,7,0
7,7.000,203,6,0,11,0
8,8.000,192,8,0,7,0
9,9.000,176,6,0,4,0


### 3.3.1 Investigating the Meaning of Historical Missing Values

The missingness analysis indicates that several historical variables are missing systematically when there is little or no previous purchase history. For example, purchase-value statistics cannot be calculated when no previous purchases exist, while measures of variability may also be undefined when only one previous purchase is available. These cases are therefore treated as structurally missing rather than automatically imputed using the overall distribution.

In [45]:
# Summarise missing historical values by previous purchase count.

historical_missing_by_count = (
    train_prep
    .groupby("prior_purchase_count", dropna=False)
    .agg(
        observations=("prior_purchase_count", "size"),
        missing_mean_purchase_value=("prior_mean_purchase_value", lambda x: x.isna().sum()),
        missing_std_purchase_value=("prior_std_purchase_value", lambda x: x.isna().sum()),
        missing_min_purchase_value=("prior_min_purchase_value", lambda x: x.isna().sum()),
        missing_max_purchase_value=("prior_max_purchase_value", lambda x: x.isna().sum()),
        missing_mean_gap_days=("prior_mean_gap_days", lambda x: x.isna().sum()),
        missing_std_gap_days=("prior_std_gap_days", lambda x: x.isna().sum())
    )
)

display(historical_missing_by_count.head(15))

,observations,missing_mean_purchase_value,missing_std_purchase_value,missing_min_purchase_value,missing_max_purchase_value,missing_mean_gap_days,missing_std_gap_days
prior_purchase_count,,,,,,,
0.000,603,603,603,603,603,603,603
1.000,445,20,445,0,0,445,445
2.000,365,11,0,0,0,25,365
3.000,319,14,0,0,0,17,0
4.000,270,11,0,0,0,7,0
5.000,248,7,0,0,0,8,0
6.000,222,6,0,0,0,7,0
7.000,203,6,0,0,0,11,0
8.000,192,8,0,0,0,7,0


In [44]:
# Display missing values for variables that are not historical statistics.

non_historical_missing = (
    missing_summary
    .drop(index=historical_columns, errors="ignore")
)

display(non_historical_missing)

,missing_count,missing_percentage
days_since_previous_purchase,812,12.552
transport_method,196,3.030
spend_365d,195,3.014
payment_type,195,3.014
warehouse,195,3.014
salesperson,194,2.999


In [47]:
# Identify observations where a historical statistic is missing
# even though previous purchases are recorded.

for col in historical_value_columns:
    suspicious = train_prep[
        (train_prep["prior_purchase_count"] > 0)
        & (train_prep[col].isna())
    ]

    print(f"{col}: {len(suspicious)} observations")

prior_mean_purchase_value: 194 observations
prior_std_purchase_value: 445 observations
prior_min_purchase_value: 0 observations
prior_max_purchase_value: 0 observations
prior_mean_gap_days: 656 observations
prior_std_gap_days: 810 observations


#### Observations

The missing-value analysis shows that several historical variables have structural missingness related to the amount of available purchase history. For example, purchase-value statistics are undefined when no previous purchases exist, while standard deviation measures require a sufficient number of previous purchases.

However, some missing values remain even when previous purchase history is available. In particular, `prior_mean_purchase_value` and `prior_mean_gap_days` contain missing observations for customers with previous purchases. These cases require an appropriate imputation strategy rather than being treated as structurally missing.

The remaining missing values will therefore be handled according to the type and meaning of each variable rather than using a single imputation rule for the entire dataset.

### 3.4 Handle Extreme Spending Values

The EDA identified strong right-skewness and extreme values in several spending-related variables. In particular, some historical spending variables contain values around €1 billion, while some purchase-value variables also contain negative values.

These observations will be investigated before deciding whether they represent valid extreme purchases, data-quality issues or values that should be transformed or capped.

In [49]:
# Spending-related variables identified during the EDA.

spending_columns = [
    "prior_total_spend",
    "prior_mean_purchase_value",
    "prior_std_purchase_value",
    "prior_min_purchase_value",
    "prior_max_purchase_value",
    "spend_30d",
    "spend_90d",
    "spend_180d",
    "spend_365d"
]

# Summarise the distributions of spending variables.

spending_summary = train_prep[spending_columns].describe().T

display(spending_summary)

,count,mean,std,min,25%,50%,75%,max
prior_total_spend,"6,469.000","18,908.349","31,130.865",0.000,"1,505.550","6,754.310","21,642.320","247,865.100"
prior_mean_purchase_value,"5,672.000","2,821,825.318","53,041,566.656","-2,478.658",359.558,724.897,"1,095.887","1,000,000,000.000"
prior_std_purchase_value,"5,421.000",737.470,666.262,0.000,258.560,569.078,981.094,"6,770.221"
prior_min_purchase_value,"5,866.000",185.462,408.399,0.930,15.450,64.340,162.610,"6,298.980"
prior_max_purchase_value,"5,866.000","2,946.378","5,901.548",3.310,834.530,"1,783.240","3,504.420","267,526.190"
spend_30d,"6,469.000",457.141,"1,191.231",0.000,0.000,0.000,338.240,"22,297.210"
spend_90d,"6,469.000","1,253.528","2,347.842",0.000,0.000,198.500,"1,514.560","30,234.880"
spend_180d,"6,469.000","2,650.600","4,020.267",0.000,149.160,"1,117.680","3,404.440","49,746.970"
spend_365d,"6,274.000","2,396,534.876","48,841,133.456","-159,115.176",637.783,"2,426.605","7,045.423","1,000,000,000.000"


In [50]:
# Count negative values in spending-related variables.

negative_spending = pd.DataFrame({
    "negative_count": (train_prep[spending_columns] < 0).sum(),
    "negative_percentage": (train_prep[spending_columns] < 0).mean() * 100
})

negative_spending = negative_spending[
    negative_spending["negative_count"] > 0
]

display(negative_spending)

,negative_count,negative_percentage
prior_mean_purchase_value,16,0.247
spend_365d,15,0.232


In [52]:
# Count extremely large spending values.

# note_ines: should we use a different threshold?

extreme_spending = pd.DataFrame({
    "values_above_1e8": (train_prep[spending_columns] > 1e8).sum(),
    "percentage_above_1e8": (train_prep[spending_columns] > 1e8).mean() * 100
})

extreme_spending = extreme_spending[
    extreme_spending["values_above_1e8"] > 0
]

display(extreme_spending)

,values_above_1e8,percentage_above_1e8
prior_mean_purchase_value,16,0.247
spend_365d,15,0.232


In [53]:
# Inspect observations with extremely large or negative spending values.

extreme_rows = train_prep[
    (train_prep["prior_mean_purchase_value"] > 1e8)
    | (train_prep["prior_mean_purchase_value"] < 0)
    | (train_prep["spend_365d"] > 1e8)
    | (train_prep["spend_365d"] < 0)
]

display(
    extreme_rows[
        [
            "ID",
            "customer_id",
            "purchase_date",
            "prior_purchase_count",
            "prior_total_spend",
            "prior_mean_purchase_value",
            "spend_365d"
        ]
    ]
)

,ID,customer_id,purchase_date,prior_purchase_count,prior_total_spend,prior_mean_purchase_value,spend_365d
14,10000018,239651,2021-09-24,31.000,"3,414.560",-110.147,"1,019.820"
196,10000226,112532,2017-08-04,9.000,"12,364.880","1,000,000,000.000",NaN
263,10000300,928271,2022-07-02,5.000,"3,456.320",691.264,"-3,456.320"
267,10000304,175924,2021-10-17,16.000,"8,663.310","1,000,000,000.000","3,094.490"
448,10000519,933227,2019-04-06,6.000,"2,458.750",-409.792,NaN
462,10000536,103940,2021-04-03,194.000,"42,343.680",-218.266,"13,067.220"
916,10001073,239651,2023-11-25,64.000,"8,467.220",132.300,"1,000,000,000.000"
991,10001157,568558,2019-03-30,7.000,"2,925.420",-417.917,567.230
1029,10001200,258332,2022-05-31,114.000,"102,909.170",902.712,"1,000,000,000.000"
1176,10001375,215423,2016-12-03,5.000,"3,895.150",-779.030,"3,261.060"


In [54]:
# Inspect the exact extreme values.

print("Extreme prior_mean_purchase_value values:")
print(
    train_prep.loc[
        (train_prep["prior_mean_purchase_value"] > 1e8)
        | (train_prep["prior_mean_purchase_value"] < 0),
        "prior_mean_purchase_value"
    ].sort_values().to_list()
)

print("\nExtreme spend_365d values:")
print(
    train_prep.loc[
        (train_prep["spend_365d"] > 1e8)
        | (train_prep["spend_365d"] < 0),
        "spend_365d"
    ].sort_values().to_list()
)

Extreme prior_mean_purchase_value values:
[-2478.6584782608693, -885.0971111111112, -883.51, -779.03, -757.4800000000001, -693.01, -504.58, -417.9171428571428, -409.7916666666667, -366.12, -357.19375, -305.93, -218.2663917525773, -212.36062200956937, -162.4823076923077, -110.14709677419351, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0]

Extreme spend_365d values:
[-159115.17553487257, -14740.289999999995, -12638.05, -6795.470000000001, -5133.039999999999, -4196.9, -3456.3199999999997, -2648.4399999999996, -2169.91, -1620.83, -1443.64, -1230.72, -753.51, -295.38, -167.18, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0, 1000000000.0]


#### Observations

The analysis identified clear anomalies in the spending variables. Both `prior_mean_purchase_value` and `spend_365d` contain negative values, which are not consistent with their interpretation as spending measures. Both variables also contain values exactly equal to €1,000,000,000, which are several orders of magnitude larger than the typical values observed in the data.

These anomalous values have a strong effect on the mean and standard deviation of the variables. Since the affected observations otherwise contain valid purchase information, the observations themselves will not be removed. Instead, the identified invalid spending values will be treated as missing and handled during the missing-value treatment.

This approach preserves the observations while preventing clearly invalid values from influencing the models.

## 3.5 Remove Uninformative Variables

Before continuing with feature engineering, variables that contain no useful
observed information or cannot provide predictive information are identified.

This step is performed using the training data only. Variables are not removed
based on their relationship with the target at this stage, since supervised
feature selection should be performed after the data has been split and using
the training portion only.

Identifier variables are also treated separately. Although they may be useful
for merging datasets or tracking observations, they should not automatically
be used as predictive features.

The same feature-removal decisions will later be applied to the test data.

### 3.5.1 Check completely missing variables

In [55]:
# Identify columns that are completely missing in the training data.

completely_missing = train_prep.columns[
    train_prep.isna().all()
].tolist()

print("Completely missing columns:")
print(completely_missing)

Completely missing columns:
[]


In [56]:
# Remove completely missing columns from training and test data.

if completely_missing:
    train_prep = train_prep.drop(columns=completely_missing)
    test_prep = test_prep.drop(columns=completely_missing)

print("Training shape:", train_prep.shape)
print("Test shape:", test_prep.shape)

Training shape: (6469, 39)
Test shape: (986, 38)


### 3.5.2 Inspect Variable Uniqueness

Variables with very few unique values may contain limited information, while
variables with almost one unique value per observation may function primarily
as identifiers.

This inspection is performed without using the target variable. Variables will
not be removed solely based on their number of unique values, since their
usefulness depends on their meaning and role in the dataset.

In [57]:
# Inspect the number of unique values in each training variable.

uniqueness_summary = pd.DataFrame({
    "unique_count": train_prep.nunique(dropna=False),
    "unique_percentage": (
        train_prep.nunique(dropna=False) / len(train_prep) * 100
    )
})

uniqueness_summary = uniqueness_summary.sort_values(
    "unique_count",
    ascending=True
)

display(uniqueness_summary)

,unique_count,unique_percentage
repeat_purchase_90d,2,0.031
document_series,3,0.046
warehouse,3,0.046
invoice_count_current_purchase,5,0.077
commercial_zone,5,0.077
payment_type,10,0.155
salesperson,10,0.155
purchase_count_30d,12,0.186
purchase_count_90d,23,0.356
purchase_count_180d,38,0.587


In [58]:
# Identify variables with only one unique value in the training data.

constant_columns = [
    col for col in train_prep.columns
    if train_prep[col].nunique(dropna=False) == 1
]

print("Constant columns:")
print(constant_columns)

Constant columns:
[]


#### Observations

No completely missing or constant variables were identified in the training
data after removing observations without purchase information.

The uniqueness analysis also confirmed that `ID` is an observation identifier
and should not be used as a predictive feature. However, it is retained for
observation tracking and later submission purposes. Similarly, `customer_id`
is retained because it is required for the integration of customer-level
information.

Variables with a small number of unique values, such as `warehouse`,
`commercial_zone`, `payment_type` and `salesperson`, were not removed because
they may represent meaningful categorical information.

The `random_noise` variable is also retained for now. Its predictive
usefulness will be evaluated during modelling using the validation procedure,
rather than removing it based on its apparent nature or its relationship with
the target before the data split.

Therefore, no variables are removed at this stage based solely on their
number of unique values or their relationship with the target.

## 3.6 Customer Data Integration

The `clientes.csv` dataset contains additional information at the customer level.
This information can potentially provide useful context for predicting repeat
purchases.

The datasets are therefore integrated using `customer_id`, which is the common
identifier between the purchase-level and customer-level data.

Before merging, the customer table is checked to ensure that `customer_id`
uniquely identifies a customer. This is important because duplicate customer
records could create multiple rows for the same purchase during the merge.

The customer information is merged without using the target variable. Only
customer attributes available independently of the current purchase outcome are
considered, helping to avoid introducing target leakage.

### 3.6.1 Check the customer table

In [59]:
# Inspect the customer-level dataset.

print("Customer dataset shape:", clientes_prep.shape)
print("\nNumber of unique customers:", clientes_prep["customer_id"].nunique())
print("\nMissing customer IDs:", clientes_prep["customer_id"].isna().sum())

Customer dataset shape: (1143, 13)

Number of unique customers: 1143

Missing customer IDs: 0


### 3.6.2 Check for duplicate customer IDs

In [60]:
# Check whether each customer_id appears only once in the customer table.

customer_id_counts = clientes_prep["customer_id"].value_counts()

duplicate_customer_ids = customer_id_counts[
    customer_id_counts > 1
]

print("Number of duplicated customer IDs:",
      len(duplicate_customer_ids))

if len(duplicate_customer_ids) > 0:
    display(duplicate_customer_ids)

Number of duplicated customer IDs: 0


### 3.6.3 Check Customer Matching

Before merging the customer-level information, the customer IDs in the purchase
data are compared with those in `clientes.csv`.

This check confirms whether the customer information is available for the
observations in the training and test datasets. The merge will use a left join
so that no purchase observations are removed if customer information is
unavailable.

In [61]:
# Check how many purchase observations have a matching customer record.

train_customer_match = train_prep["customer_id"].isin(
    clientes_prep["customer_id"]
)

test_customer_match = test_prep["customer_id"].isin(
    clientes_prep["customer_id"]
)

print("Training observations with matching customer:", train_customer_match.sum())
print("Training observations without matching customer:", (~train_customer_match).sum())

print("\nTest observations with matching customer:", test_customer_match.sum())
print("Test observations without matching customer:", (~test_customer_match).sum())

Training observations with matching customer: 6469
Training observations without matching customer: 0

Test observations with matching customer: 986
Test observations without matching customer: 0


### 3.6.4 Merge Customer Information

Since all customer IDs in both the training and test datasets have a matching
record in `clientes.csv`, the customer-level information can be integrated
using a left join.

The merge is performed separately for training and test data and does not use
the target variable. After merging, the number of observations is checked to
ensure that no rows were duplicated or removed.

In [62]:
# Merge customer-level information into the purchase datasets.

train_prep = train_prep.merge(
    clientes_prep,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

test_prep = test_prep.merge(
    clientes_prep,
    on="customer_id",
    how="left",
    validate="many_to_one"
)

print("Training shape after merge:", train_prep.shape)
print("Test shape after merge:", test_prep.shape)

Training shape after merge: (6469, 51)
Test shape after merge: (986, 50)


In [63]:
# Verify that the merge did not change the number of observations.

print("Training rows:", len(train_prep))
print("Test rows:", len(test_prep))

print("\nExpected training rows:", 6469)
print("Expected test rows:", 986)

Training rows: 6469
Test rows: 986

Expected training rows: 6469
Expected test rows: 986


#### Observations

The customer-level information was successfully merged using `customer_id`.

All training and test observations had a corresponding customer record, and
the number of observations remained unchanged after the merge. This confirms
that the merge did not introduce duplicated or unmatched purchase records.

The resulting datasets contain 51 columns in training and 50 columns in test,
including the additional customer-level variables.

### 3.6.5 Inspect Customer Variables

After merging the customer-level information, the added variables are inspected
before deciding which ones should be retained for modelling.

The inspection focuses on data types, missing values and the number of unique
values. This helps identify variables that may require further preprocessing,
have limited information, or behave like identifiers.

In [64]:
# Identify the variables added from the customer table.
customer_columns = [
    col for col in clientes_prep.columns
    if col != "customer_id"
]

# Inspect their data types, missing values and number of unique values.
customer_summary = pd.DataFrame({
    "data_type": train_prep[customer_columns].dtypes,
    "missing_count": train_prep[customer_columns].isna().sum(),
    "missing_percentage": train_prep[customer_columns].isna().mean() * 100,
    "unique_count": train_prep[customer_columns].nunique(),
    "unique_percentage": (
        train_prep[customer_columns].nunique() / len(train_prep) * 100
    )
})

display(customer_summary.sort_values("unique_count"))

,data_type,missing_count,missing_percentage,unique_count,unique_percentage
customer_type,float64,6469,100.000,0,0.000
country_or_region,str,0,0.000,7,0.108
island,str,2189,33.838,10,0.155
customer_since_year,int64,0,0.000,10,0.155
commercial_region,str,35,0.541,32,0.495
municipality,str,217,3.354,150,2.319
postal_area,str,278,4.297,188,2.906
city,str,2128,32.895,259,4.004
street,str,3808,58.865,264,4.081
postal_code,str,223,3.447,516,7.977


#### Observations

The customer table contains useful geographic and customer-history information,
but several variables have substantial missingness or a high number of unique
values.

`customer_type` is completely missing and therefore provides no information.
`address` and `company_name` have a high number of unique values and may behave
as identifier-like variables rather than generalizable customer characteristics.
`street` also has substantial missingness.

The remaining customer variables are retained for now and will be evaluated
during encoding and feature selection. This avoids removing potentially useful
information based only on missingness or cardinality.

### 3.6.6 Remove Uninformative Customer Variables

Based on the previous inspection, `customer_type` is completely missing and
therefore cannot provide useful information for modelling.

`address` and `company_name` have a very high number of unique values relative
to the number of observations and may behave as identifier-like variables.
These variables are therefore removed to reduce the risk of the model learning
customer-specific patterns that are unlikely to generalize.

The remaining customer variables are retained for further preprocessing and
feature selection.

In [65]:
# Remove customer variables that provide no observed information
# or are likely to behave as identifiers.

customer_columns_to_drop = [
    "customer_type",
    "address",
    "company_name"
]

train_prep = train_prep.drop(columns=customer_columns_to_drop)
test_prep = test_prep.drop(columns=customer_columns_to_drop)

print("Removed customer variables:", customer_columns_to_drop)
print("Training shape:", train_prep.shape)
print("Test shape:", test_prep.shape)

Removed customer variables: ['customer_type', 'address', 'company_name']
Training shape: (6469, 48)
Test shape: (986, 47)


#### Observations

Three customer-level variables were removed because they were either completely
missing or likely to behave as identifier-like variables. The remaining
customer variables are retained for further preprocessing and feature
selection.

### 3.6.7 Inspect Missing Customer Information

The remaining customer-level variables contain different amounts of missing
information. Rather than applying a single rule to all variables, their
missingness is inspected before selecting an appropriate treatment.

Categorical variables with missing values may require an explicit missing
category, while variables with substantial missingness may need to be
reconsidered during feature selection.

In [67]:
# Identify the customer variables that are still present after the previous cleaning step.
customer_columns = [
    col for col in clientes_prep.columns
    if col != "customer_id"
    and col in train_prep.columns
]

# Summarise missing values in the remaining customer variables.
customer_missing_summary = pd.DataFrame({
    "missing_count": train_prep[customer_columns].isna().sum(),
    "missing_percentage": train_prep[customer_columns].isna().mean() * 100
})

customer_missing_summary = (
    customer_missing_summary[
        customer_missing_summary["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
)

display(customer_missing_summary)

,missing_count,missing_percentage
street,3808,58.865
island,2189,33.838
city,2128,32.895
postal_area,278,4.297
postal_code,223,3.447
municipality,217,3.354
commercial_region,35,0.541


In [68]:
# Inspect the number of observations with and without street information.

street_missing = train_prep["street"].isna().sum()
street_present = train_prep["street"].notna().sum()

print("Street information available:", street_present)
print("Street information missing:", street_missing)
print(
    "Percentage missing:",
    round(street_missing / len(train_prep) * 100, 2),
)

Street information available: 2661
Street information missing: 3808
Percentage missing: 58.87


In [69]:
# Check whether street information is available for most customers
# or whether it is frequently missing for individual customers.

street_by_customer = (
    train_prep.groupby("customer_id")["street"]
    .agg(
        observations="size",
        missing_street=lambda x: x.isna().sum()
    )
)

street_by_customer["missing_percentage"] = (
    street_by_customer["missing_street"]
    / street_by_customer["observations"]
    * 100
)

display(
    street_by_customer
    .sort_values("missing_percentage", ascending=False)
    .head(10)
)

,observations,missing_street,missing_percentage
customer_id,,,
103940,276,276,100.000
385169,2,2,100.000
630805,2,2,100.000
630646,2,2,100.000
625161,62,62,100.000
615117,1,1,100.000
613781,7,7,100.000
600779,5,5,100.000
600547,1,1,100.000


#### Observations

The `street` variable is missing in 58.87% of the training observations.
The missingness is also frequently consistent within individual customers,
with several customers having no street information across all of their
observations.

Given its high missingness and relatively granular nature, `street` is
removed rather than imputed. The remaining geographic variables provide less
sparse alternatives and will be evaluated further.

In [70]:
# Remove the street variable due to its high and systematic missingness.

train_prep = train_prep.drop(columns=["street"])
test_prep = test_prep.drop(columns=["street"])

print("Removed variable: street")
print("Training shape:", train_prep.shape)
print("Test shape:", test_prep.shape)

Removed variable: street
Training shape: (6469, 47)
Test shape: (986, 46)


### 3.6.8 Inspect Remaining Customer Missing Values

After removing `street`, the remaining customer-level variables have relatively
low to moderate levels of missingness.

For categorical variables, missing values can potentially represent the absence
of available information rather than a specific category. Therefore, the
remaining missing values will be handled explicitly during the categorical
preprocessing stage rather than being replaced with an arbitrary existing
category.

In [72]:
customer_columns = [
    col for col in clientes_prep.columns
    if col != "customer_id"
    and col in train_prep.columns
]

# Inspect the remaining customer variables with missing values.

customer_missing_summary = pd.DataFrame({
    "missing_count": train_prep[customer_columns].isna().sum(),
    "missing_percentage": train_prep[customer_columns].isna().mean() * 100
})

customer_missing_summary = (
    customer_missing_summary[
        customer_missing_summary["missing_count"] > 0
    ]
    .sort_values("missing_count", ascending=False)
)

display(customer_missing_summary)

,missing_count,missing_percentage
island,2189,33.838
city,2128,32.895
postal_area,278,4.297
postal_code,223,3.447
municipality,217,3.354
commercial_region,35,0.541


In [73]:
# Check whether missing geographic information is consistent within customers.

geographic_columns = [
    "country_or_region",
    "island",
    "commercial_region",
    "municipality",
    "postal_area",
    "city",
    "postal_code"
]

customer_missingness = []

for col in geographic_columns:
    customer_status = (
        train_prep.groupby("customer_id")[col]
        .agg(
            observations="size",
            missing_count=lambda x: x.isna().sum()
        )
    )

    customer_status["missing_percentage"] = (
        customer_status["missing_count"]
        / customer_status["observations"]
        * 100
    )

    customer_missingness.append({
        "variable": col,
        "customers_with_missing": (
            customer_status["missing_count"] > 0
        ).sum(),
        "customers_100pct_missing": (
            customer_status["missing_percentage"] == 100
        ).sum()
    })

customer_missingness = pd.DataFrame(customer_missingness)

display(customer_missingness)

,variable,customers_with_missing,customers_100pct_missing
0,country_or_region,0,0
1,island,284,284
2,commercial_region,12,12
3,municipality,41,41
4,postal_area,40,40
5,city,234,234
6,postal_code,20,20


#### Observations

The missingness analysis shows that, for all customer-level geographic variables
with missing values, the number of customers with missing information is equal
to the number of customers for whom the variable is missing across all
observations.

This indicates that the missingness is consistent at customer level rather than
occurring randomly across individual purchases. Therefore, missing values in
these categorical variables will be treated as an explicit `"Missing"`
category rather than being replaced by an existing category.

### 3.6.10 Handle Missing Customer Categories

The customer-level missingness analysis showed that missing geographic
information is consistent within customers. When a geographic variable is
missing, it is missing across all observations for that customer.

Therefore, missing values in the remaining categorical customer variables are
treated as an explicit `"Missing"` category rather than being replaced by an
existing category. This preserves the distinction between known information
and information that is unavailable.

This transformation does not require estimating values from the data and is
therefore applied consistently to both training and test data.

In [74]:
# Replace missing values in the remaining categorical customer variables
# with an explicit "Missing" category.

categorical_customer_columns = [
    "country_or_region",
    "island",
    "commercial_region",
    "municipality",
    "postal_area",
    "city",
    "postal_code"
]

for col in categorical_customer_columns:
    train_prep[col] = train_prep[col].fillna("Missing")
    test_prep[col] = test_prep[col].fillna("Missing")

# Verify that no missing values remain in these customer variables.
customer_missing_after = pd.DataFrame({
    "missing_count_train": train_prep[categorical_customer_columns].isna().sum(),
    "missing_count_test": test_prep[categorical_customer_columns].isna().sum()
})

display(customer_missing_after)

,missing_count_train,missing_count_test
country_or_region,0,0
island,0,0
commercial_region,0,0
municipality,0,0
postal_area,0,0
city,0,0
postal_code,0,0


#### Observations

All missing values in the remaining categorical customer variables were
successfully replaced by the explicit `"Missing"` category in both training and
test data. No missing values remain in these variables.